# W16-D4 实验：两套锚点周验预演 × 验证器突变测试

**与 md 的分工**：md（`第16周-Day4-锚点周验预演×Identity验证器补位×G01证据链深化.md`）是阅读材料；本 ipynb 是**可执行证据**——两套锚点门实跑（G-05 × 15 + Identity 26 项）、子串碰撞假 OK 复现（朴素匹配 vs 词边界）、注入突变排练（M1-M4 + G-05 突变，验证器「能红」的证明）、突变检出矩阵可视化。

**Today's Question**：昨天的验收标准是「红绿可测」，今晨 41 项全绿——那验证器自己错了谁来红？为什么全绿之后还要专门排练周验？

工件：`governance/ci/identity_anchor_ci.py`（今日补位）+ `governance/identity-analysis-anchors.yaml` v0.1.1（负锚点重锚）+ `sync/s1-baseline-checklist.yaml`（rehearsal 记录）+ `changes/g01-ontology-governance-header/evidence/claims-2026-09-17.json`。

In [ ]:
# ---- 环境：中文字体（TOOLS.md 标准方式）+ 路径 ----
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

import json, re, subprocess, sys, tempfile, yaml, datetime
from pathlib import Path

BASE = Path("/root/learning-notebooks")
SEM = BASE / "semantic-model"
GOV = SEM / "governance"
W16 = BASE / "第16周"
LNKCRE = Path("/root/lnkcre")

def run_gate(script, *args):
    """跑真实门禁脚本，返回 (exit_code, stdout)。"""
    r = subprocess.run([sys.executable, str(script), *args], capture_output=True, text=True)
    return r.returncode, r.stdout + r.stderr

print("S2 日探针（2026-09-17 晨间实测）: lnkcre behind=0（HEAD 0392e107）| docs behind=15（c3d08d6，10→15 连续累积）| LnkChatBI behind=0")
print("挡板判定: max 15 << 300 → 未触发，W16 批次维持")

## §1 周验预演第一跑：两套锚点门实跑（真实仓库、真实登记）

In [ ]:
# ---- G-05 anchors 门：15 锚 × lnkcre HEAD 0392e107（基线同头 → 期望全绿）----
rc_g05, out_g05 = run_gate(GOV / "ci/frozen_effect_ci.py", "anchors",
                           "--anchors", str(GOV / "g05-effect-anchors.yaml"), "--repo", str(LNKCRE))
print(out_g05.strip().splitlines()[-1])
print(f"G-05 anchors 门 exit={rc_g05}")
assert rc_g05 == 0, "G-05 门应绿（基线头未动）"

# ---- Identity 门：修复后登记 v0.1.1 × 真实 spec（期望 26/26 绿）----
rc_id, out_id = run_gate(GOV / "ci/identity_anchor_ci.py", "verify",
                         "--anchors", str(GOV / "identity-analysis-anchors.yaml"), "--repo", str(LNKCRE))
print(out_id.strip().splitlines()[0])
print(out_id.strip().splitlines()[-1])
print(f"Identity verify 门 exit={rc_id}")
assert rc_id == 0, "Identity 门（重锚修复后）应绿"

n_anchor_total = 15 + 26
print(f"\n周验预演基线：G-05 {15} 锚 + Identity {26} 项 = {n_anchor_total} 项全绿（W39 digest 可执行性确认）")

## §2 碰撞复现：朴素子串匹配的「假 OK」是怎么发生的

D3 登记的负锚点 `snap_lease_daily` 注册在 L17——那一行只有授权对象 `gov_snap_lease_daily`。朴素子串匹配（`expect in line`，G-05 门同款）会判 OK，**锚的是错误证据**。词边界正则 `(?<![A-Za-z0-9_])name(?![A-Za-z0-9_])` 不受前缀 `gov_` 干扰。

In [ ]:
# ---- 用真实 spec 行做匹配器对照实验 ----
spec_lines = (LNKCRE / "openspec/specs/chatbi-governed-account-boundary/spec.md").read_text().splitlines()
L17, L23 = spec_lines[16], spec_lines[22]
print("L17（授权对象行）:", L17.strip())
print("L23（排除条款行）:", L23.strip())

def substring_match(name, line): return name in line
def word_boundary(name):
    return re.compile(r"(?<![A-Za-z0-9_])" + re.escape(name) + r"(?![A-Za-z0-9_])")

cases = [
    ("负锚点×L17（gov_ 授权行，错误证据）", "snap_lease_daily", L17),
    ("负锚点×L23（排除条款，正确证据）",   "snap_lease_daily", L23),
    ("正锚点×L17（gov_snap_lease_daily 本尊）", "gov_snap_lease_daily", L17),
]
print(f"\n{'场景':<32s} {'朴素子串':>8s} {'词边界':>8s}  判读")
collision_rows = []
for desc, name, line in cases:
    s, w = substring_match(name, line), bool(word_boundary(name).search(line))
    verdict = "子串假 OK！" if (s and not w) else ("双绿（正锚点场景两法等价）" if s == w else "子串漏检")
    print(f"{desc:<34s}{'命中' if s else '未中':>7s}{'命中' if w else '未中':>8s}  {verdict}")
    collision_rows.append((desc, s, w))

# 关键断言：朴素子串在 L17 命中（假 OK），词边界不命中；L23 两者都命中（真实证据）
assert collision_rows[0][1] and not collision_rows[0][2], "子串应在 L17 假命中、词边界不命中"
assert collision_rows[1][1] and collision_rows[1][2], "L23 是两法一致的真实证据行"
print("\n结论：负锚点若用朴素子串 + L17 注册，会『绿着锚错证据』一整天——预演第一跑的 DRIFTED 判决把它暴露（词边界真实命中在 L23 → 重锚修复 v0.1.1）")

## §3 注入突变排练：验证器「能红」的证明（identity selftest + G-05 突变）

**消防演习逻辑**：平时烧一场假火，火警真响才敢信半夜真着火时它不睡岗。4 类注入突变 + 1 碰撞哨兵（identity），1 类代码突变（G-05，临时仓副本——只拷贝锚定文件，改一个 `StatusDraft` → 期望恰好 1 BROKEN 红门）。

In [ ]:
# ---- Identity selftest：4 突变 + 碰撞哨兵（真实脚本、临时仓注入）----
rc_st, out_st = run_gate(GOV / "ci/identity_anchor_ci.py", "selftest",
                         "--anchors", str(GOV / "identity-analysis-anchors.yaml"), "--repo", str(LNKCRE))
print(out_st.strip())
assert rc_st == 0, "selftest 应全 PASS（验证器有牙齿）"

# ---- G-05 突变：完整锚定文件副本 + 单点突变 → 期望恰好 1 BROKEN → exit 1 ----
anchors = yaml.safe_load((GOV / "g05-effect-anchors.yaml").read_text())
anchored_files = sorted({a["file"] for spec in anchors["effects"].values() for a in spec["anchors"]})
with tempfile.TemporaryDirectory() as td:
    repo = Path(td)
    for rel in anchored_files:  # 拷贝全部 14 个锚定文件 → 突变后其余锚点仍可对账
        dst = repo / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        dst.write_text((LNKCRE / rel).read_text(), encoding="utf-8")
    mutated = repo / "backend/internal/lease/model.go"
    mutated.write_text(mutated.read_text().replace("StatusDraft", "StatusMutated", 1), encoding="utf-8")
    rc_g05m, out_g05m = run_gate(GOV / "ci/frozen_effect_ci.py", "anchors",
                                 "--anchors", str(GOV / "g05-effect-anchors.yaml"), "--repo", str(repo))
    broken_line = [l for l in out_g05m.splitlines() if "BROKEN" in l and l.startswith("[")]  # 排除合计行
    print("\nG-05 突变（临时仓 14 文件副本 + StatusDraft→StatusMutated）:")
    print(broken_line[0] if broken_line else "(未检出！)")
    print(out_g05m.strip().splitlines()[-1], f"→ exit={rc_g05m}")
    assert rc_g05m == 1 and len(broken_line) == 1, "应恰好 1 个 BROKEN 且门红"

print("\n排练结论：两套门对各自最危险的漂移方向（Identity=授权区泄漏/排除消失；G-05=冻结证据改写）都有『能红』的机器证据")

## §4 可视化：突变检出矩阵（谁的火警响了）

In [ ]:
# ---- 突变 × 检出方式矩阵：红=必红门, 黄=DRIFTED(需 --strict 才红), 绿=按设计不红 ----
import numpy as np

mutations = ["M1 删正锚点\n(授权对象消失)", "M2 删排除条款\n(restricted 提及被抹)",
             "M3 授权区泄漏\n(restricted 入授权段)", "M4 行号漂移\n(插一行全体下移)",
             "碰撞场景\n(gov_x 行 vs 负锚点 x)", "G-05 冻结证据改写\n(StatusDraft→Mutated)"]
detectors = ["朴素子串匹配\n(D3 假想验证器)", "词边界+泄漏检查\n(identity_anchor_ci)", "G-05 anchors 门\n(子串,Go 代码锚)"]
# 值: 2=红(检出), 1=黄(DRIFTED), 0=未检出/不适用; None=N/A
M = np.array([
    #         M1   M2   M3   M4   碰撞  G05改写
    [          2,   2,   1,   1,    0,    2],  # 朴素子串: M3 只黄(排除条款还在,泄漏不可见), M4 黄, 碰撞=0(假OK)
    [          2,   2,   2,   1,    2,   -1],  # 词边界门: M3 红(泄漏检查), 碰撞红(哨兵), G05=N/A
    [         -1,  -1,   0,   1,   -1,    2],  # G-05 门: 只锚 Go 代码, M3 对它无对象, 冻结改写红
], dtype=float)

fig, ax = plt.subplots(figsize=(9.5, 4.2))
from matplotlib.colors import ListedColormap
cmap = ListedColormap(["#c9ccd4", "#e8b93d", "#d64541"])  # 灰=未检出/N-A, 黄=DRIFTED, 红=红门检出
im = ax.imshow(M, cmap=cmap, vmin=0, vmax=2, aspect="auto")
ax.set_xticks(range(len(mutations)), mutations, fontsize=8.5)
ax.set_yticks(range(len(detectors)), detectors, fontsize=9)
label = {2: "红", 1: "黄", 0: "漏", -1: "N/A"}
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        color = "white" if M[i, j] in (2, 0) else "black"
        ax.text(j, i, label[int(M[i, j])], ha="center", va="center", fontsize=11, color=color, fontweight="bold")
ax.set_title("W16-D4 突变检出矩阵：验证器『能红』才是资质，『当前绿』不是\n(实测: Identity selftest 5/5 PASS + G-05 突变 exit=1 恰 1 BROKEN)", fontsize=10.5)
fig.tight_layout()
fig.savefig(W16 / "w16d4_mutation_matrix.png", dpi=150)
print("已保存 w16d4_mutation_matrix.png")
plt.show()

# 矩阵断言：朴素子串在碰撞场景漏检(0)、词边界门 M3 红(2)、G-05 门冻结改写红(2)
assert M[0, 4] == 0 and M[1, 2] == 2 and M[2, 5] == 2

## §5 G-01 证据链复算（受理状态核查的机器面）

In [ ]:
# ---- 复算 evidence/claims-2026-09-17.json 的 C1/C3：受理核查不是转述，是复算 ----
import hashlib
def sha16(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:16]

docs = Path("/root/docs")
sot = docs / "lanlnk/config/ontology/business-ontology.yaml"
claims = json.loads((SEM / "changes/g01-ontology-governance-header/evidence/claims-2026-09-17.json").read_text())
c1 = claims["evidence_chain"]["claims"][0]
fp_now = sha16(sot)
print(f"C1 SoT 指纹复算: 今日实测 {fp_now} | 证据件登记 {c1['actual']} | 一致={fp_now == c1['actual']}")
assert fp_now == c1["actual"] == "bf550bc24de66813"

family = {
    "business-ontology":   "lanlnk/config/ontology/business-ontology.yaml",
    "lnkchat-30products":  "lanlnk/30-products/lnkchat/ontology.yaml",
    "LnkChatBI-out":       "lanlnk/out/prd/LnkChatBI/output/ontology.yaml",
    "lnkreport-out":       "lanlnk/out/prd/lnkreport/output/ontology.yaml",
    "legacy-frozen":       "lanlnk/90-legacy/2026-07-31-pre-document-control-plane/out-prd-langchat/output/ontology.yaml",
}
c2 = claims["evidence_chain"]["claims"][1]
drift = {k: sha16(docs / p) for k, p in family.items()}
expect2 = {("legacy-frozen" if k == "legacy-frozen-snapshot" else k): v for k, v in c2["expected"].items()}  # JSON 键名→family 命名
same = {k: (drift[k] == expect2[k]) for k in family}  # JSON 键名 legacy-frozen-snapshot
print(f"C2 五件家族指纹复算（docs +15 commits 后）: 一致 {sum(same.values())}/5")
for k in family:
    print(f"  {k:<20s} {drift[k]}  {'PASS' if same[k] else 'DRIFTED!'}")
assert all(same.values()), "五件指纹应与 9/14 审计一致"
print("\n受理状态（C4）: docs 增量 15 commits 零 ontology/governance 主题 → 立案窗口未开启（对账侧五断言全 PASS，受理裁决权在主仓）")

## §6 结论：Today's Question 的回答

**验证器自己错了谁来红？没有人——除非专门造一个会错的场景让它红给你看。**「全绿」只证明验证器对今天的输入没有意见，不证明验证器对：负锚点 `snap_lease_daily` 用朴素子串在 `gov_snap_lease_daily`（授权对象）上**假 OK 绿了一整天**，若今日不排练，它会绿到 W39、绿到白名单真被改坏的那天。注入排练就是消防演习：M1/M2/M3 三类真危险全部 exit 1（能红），M4 行号漂移黄不红（不误伤），碰撞哨兵守住回归（防再犯）。

方法论沉淀：**门禁交付必须附带突变证据（改 X 必红、不改必绿、改无关不误伤）——没有突变证据的门禁只是绿色的装饰品。** brief ② 验收线今日从「红绿可测」升级为「红绿可测 + 突变留痕」（`identity_anchor_ci.py selftest` 成为可重放子命令）。

**验证汇总**：G-05 门 15/15 OK（GREEN，HEAD 0392e107 零漂移）｜Identity 门修复后 26/26 OK（20 正 + 6 负 + 授权区 L10-L18 零泄漏）｜selftest 5/5（M1-M3 exit 1、M4 exit 0、碰撞哨兵 PASS）｜G-05 突变恰 1 BROKEN exit 1｜G-01 证据链 C1/C2 复算 PASS（五件指纹零漂移）+ C4 未受理确认｜W39 digest 五项检查全部可执行（格式问题 2 项当日清零：负锚点无验证器、子串碰撞假 OK）。

**明日（W16-D5）**：brief 验收对账预演（①-④ 验收线逐条自评 + 缺口清单）；W17 backlog 固化（G-07 新表挡板 citing-action 范本草稿——抄 Identity 三件套结构：正锚点 + 负锚点 + 泄漏检查）；周日 Review 材料包（五维评分 + 同步健康增查）。